# End-to-End Data Ingestion and Processing Pipeline (AWS MLOps)

## Environment Setup & Imports

In [ ]:
import boto3
import pandas as pd
import sagemaker
from pyathena import connect
import warnings
warnings.filterwarnings("ignore")
import pyathena
print(pyathena.__version__)

import pyarrow
import s3fs
print(pyarrow.__version__)
print(s3fs.__version__)

## Aws Configuration
### S3 Bucket and Paths

In [16]:
sess = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

bucket = "ads508-team1-sephora"   
database_name = "sephora_reviews_db"

s3_staging_dir = f"s3://{bucket}/athena/staging/"
raw_products_dir = f"s3://{bucket}/raw/products/"
raw_reviews_dir = f"s3://{bucket}/raw/reviews/"

print(region)
print(role)
print(s3_staging_dir)
print(raw_products_dir)
print(raw_reviews_dir)

us-east-1
arn:aws:iam::471590717821:role/LabRole
s3://ads508-team1-sephora/athena/staging/
s3://ads508-team1-sephora/raw/products/
s3://ads508-team1-sephora/raw/reviews/


## Data Ingestion from S3
### Load product from raw

In [ ]:
df = pd.read_csv("s3://ads508-team1-sephora/raw/products/product_info.csv")
print(df.columns.tolist())

['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']


### Load Review from raw

In [ ]:
df = pd.read_csv("s3://ads508-team1-sephora/raw/reviews/reviews_1250-end.csv")
print(df.columns.tolist())

['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']


## Data Processing & Combinng
### Combine Review Files

In [8]:
# Combine review data
review_files = [
"s3://ads508-team1-sephora/raw/reviews/reviews_0-250.csv",
"s3://ads508-team1-sephora/raw/reviews/reviews_250-500.csv",
"s3://ads508-team1-sephora/raw/reviews/reviews_500-750.csv",
"s3://ads508-team1-sephora/raw/reviews/reviews_750-1250.csv",
"s3://ads508-team1-sephora/raw/reviews/reviews_1250-end.csv"
]

dfs = []

for f in review_files:
    df = pd.read_csv(f)
    dfs.append(df)

reviews = pd.concat(dfs, ignore_index=True)

print("Total reviews:", reviews.shape)

Total reviews: (1094411, 19)


### Remove Unnecessary Index Columns

In [ ]:
# Remove index from reviews
reviews = reviews.loc[:, ~reviews.columns.str.contains("^Unnamed")]
# check schema of the combined data
reviews.columns

Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')

In [ ]:
# check the combined data
reviews.head()

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


## Data Transformation ( Curated Layer)
### Convert to Parquet Format

In [11]:
products = pd.read_csv("s3://ads508-team1-sephora/raw/products/product_info.csv")

print(products.shape)
products.head()

(8494, 27)


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


In [12]:
reviews_output = "s3://ads508-team1-sephora/curated/reviews/"
products_output = "s3://ads508-team1-sephora/curated/products/"

### Save parquet to S3 Curated Layer

In [14]:
# align author_id data type to str(int, str, float)
reviews["author_id"] = reviews["author_id"].astype(str)

reviews.to_parquet(
reviews_output + "reviews.parquet",
engine="pyarrow",
index=False
)

products.to_parquet(
products_output + "products.parquet",
engine="pyarrow",
index=False
)

## Athena Intergration

In [27]:
from pyathena import connect

# Connect to Athena
conn = connect(
    s3_staging_dir="s3://ads508-team1-sephora/athena/staging/",
    region_name="us-east-1"
)

cursor = conn.cursor()

# Create reviews table
cursor.execute("""
CREATE EXTERNAL TABLE IF NOT EXISTS sephora_reviews (
    author_id string,
    rating int,
    is_recommended boolean,
    helpfulness double,
    total_feedback_count int,
    total_neg_feedback_count int,
    total_pos_feedback_count int,
    submission_time string,
    review_text string,
    review_title string,
    skin_tone string,
    eye_color string,
    skin_type string,
    hair_color string,
    product_id string,
    product_name string,
    brand_name string,
    price_usd double
)
STORED AS PARQUET
LOCATION 's3://ads508-team1-sephora/curated/reviews/';
""")

In [28]:
# Create products table
cursor.execute("""
CREATE EXTERNAL TABLE IF NOT EXISTS sephora_products (
    product_id string,
    product_name string,
    brand_id string,
    brand_name string,
    loves_count int,
    rating double,
    reviews int,
    size string,
    variation_type string,
    variation_value string,
    variation_desc string,
    ingredients string,
    price_usd double,
    value_price_usd double,
    sale_price_usd double,
    limited_edition boolean,
    new boolean,
    online_only boolean,
    out_of_stock boolean,
    sephora_exclusive boolean,
    highlights string,
    primary_category string,
    secondary_category string,
    tertiary_category string,
    child_count int,
    child_max_price double,
    child_min_price double
)
STORED AS PARQUET
LOCATION 's3://ads508-team1-sephora/curated/products/';
""")


# Exploratory Data Analysis (EDA)
## Data statistic overview
### Data Shape

In [31]:
print("Reviews Dataset")
print("Number of Rows:", reviews.shape[0])
print("Number of Columns:", reviews.shape[1])

print()

print("Products Dataset")
print("Number of Rows:", products.shape[0])
print("Number of Columns:", products.shape[1])

Reviews Dataset
Number of Rows: 1094411
Number of Columns: 18

Products Dataset
Number of Rows: 8494
Number of Columns: 27


In [ ]:
# Schema Inspection
reviews.info()
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 18 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   author_id                 1094411 non-null  object 
 1   rating                    1094411 non-null  int64  
 2   is_recommended            926423 non-null   float64
 3   helpfulness               532819 non-null   float64
 4   total_feedback_count      1094411 non-null  int64  
 5   total_neg_feedback_count  1094411 non-null  int64  
 6   total_pos_feedback_count  1094411 non-null  int64  
 7   submission_time           1094411 non-null  object 
 8   review_text               1092967 non-null  object 
 9   review_title              783757 non-null   object 
 10  skin_tone                 923872 non-null   object 
 11  eye_color                 884783 non-null   object 
 12  skin_type                 982854 non-null   object 
 13  hair_color                8

### Summary Statistics

In [ ]:
# Reviews Summary

def eda_summary(df):

    summary = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.values,
        "Missing Values": df.isnull().sum().values,
        "Missing %": (df.isnull().mean()*100).round(2).values,
        "Unique Values": df.nunique().values,
        "Sample Values": df.apply(lambda x: list(x.dropna().unique()[:3])).values
    })

    return summary

reviews_summary = eda_summary(reviews)
reviews_summary


,Column,Data Type,Missing Values,Missing %,Unique Values,Sample Values
0,author_id,object,0,0.00,503216,"[1741593524, 31423088263, 5061282401]"
1,rating,int64,0,0.00,5,"[5, 1, 4]"
2,is_recommended,float64,167988,15.35,2,"[1.0, 0.0]"
3,helpfulness,float64,561592,51.31,3767,"[1.0, 0.25, 0.4444440007209778]"
4,total_feedback_count,int64,0,0.00,676,"[2, 0, 1]"
5,total_neg_feedback_count,int64,0,0.00,259,"[0, 6, 5]"
6,total_pos_feedback_count,int64,0,0.00,590,"[2, 0, 1]"
7,submission_time,object,0,0.00,5317,"[2023-02-01, 2023-03-21, 2023-03-20]"
8,review_text,object,1444,0.13,969419,[I use this with the Nudestix “Citrus Clean Ba...
9,review_title,object,310654,28.39,364105,"[Taught me how to double cleanse!, Disappointe..."


In [ ]:
# Products summary
products_summary = eda_summary(products)
products_summary

,Column,Data Type,Missing Values,Missing %,Unique Values,Sample Values
0,product_id,object,0,0.00,8494,"[P473671, P473668, P473662]"
1,product_name,object,0,0.00,8415,"[Fragrance Discovery Set, La Habana Eau de Par..."
2,brand_id,int64,0,0.00,304,"[6342, 6471, 6485]"
3,brand_name,object,0,0.00,304,"[19-69, 54 Thrones, ABBOTT]"
4,loves_count,int64,0,0.00,7436,"[6320, 3827, 3253]"
5,rating,float64,278,3.27,4394,"[3.6364, 4.1538, 4.25]"
6,reviews,float64,278,3.27,1556,"[11.0, 13.0, 16.0]"
7,size,object,1631,19.20,2055,"[3.4 oz/ 100 mL, 0.25 oz/ 7.5 mL, 6 oz / 180 mL]"
8,variation_type,object,1444,17.00,7,"[Size + Concentration + Formulation, Scent, Size]"
9,variation_value,object,1598,18.81,2729,"[3.4 oz/ 100 mL, 0.25 oz/ 7.5 mL Eau de Parfum..."


# Pipeline Summary
This notebook implements an end-to-end data ingestion and transformation pipeline:
- Raw → Curated (S3)
- CSV → Parquet
- Athena-ready datasets